<a href="https://colab.research.google.com/github/Machine2026/Sem.10-Inteligencia-Artificial-Rob-tica/blob/main/Semana_10_AI_Robotica_Fundamentos_Inferencia_Colab_ES.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Semana 10 · Inteligencia Artificial & Robótica
## De una neurona artificial a modelos abiertos pequeños: fundamentos, inferencia y validación de datos

**Curso:** Inteligencia Artificial & Robótica · ESAN · V ciclo  
**Modalidad de esta sesión:** Google Colab · clase online · sin laboratorio físico ni tarjetas RTX locales  
**Propósito:** entender, ejecutando código, cómo la IA evoluciona desde una unidad computacional simple hasta modelos generativos pequeños.

### Qué construirás hoy
1. Un **perceptrón** desde cero.
2. Una **neurona logística entrenable** con función de pérdida y gradiente.
3. Una **red neuronal pequeña** que resuelve XOR.
4. Un **mini-modelo generativo** de siguiente carácter entrenado con corpus industrial.
5. Una **primera inferencia** con un modelo abierto pequeño sin fine-tuning.
6. Un **validador de formato de datos** para tu caso industrial.

> Recomendación: Colab CPU es suficiente para las secciones 1 a 5. Para el modelo abierto pequeño, GPU ayuda, pero no es estrictamente obligatoria.

In [1]:
# Celda 1 · Verificación del entorno
import sys, math, random, json, os, time, textwrap, re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA disponible:", torch.cuda.is_available())
    print("Dispositivo:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
except Exception as e:
    print("PyTorch todavía no está cargado:", e)

print("Python:", sys.version.split()[0])
print("Entorno listo para comenzar.")

PyTorch: 2.11.0+cpu
CUDA disponible: False
Dispositivo: CPU
Python: 3.12.13
Entorno listo para comenzar.


## 1. El perceptrón: una neurona artificial mínima

Un perceptrón recibe entradas numéricas, las multiplica por pesos, suma un sesgo y aplica una regla de decisión.

\[
z = w_1x_1 + w_2x_2 + \cdots + w_nx_n + b
\]

\[
\hat{y} = \begin{cases}
1 & \text{si } z \ge 0 \\
0 & \text{si } z < 0
\end{cases}
\]

Interpretación técnica:

- **Entradas \(x\):** señales, mediciones o atributos de un caso.
- **Pesos \(w\):** importancia y dirección de cada entrada.
- **Sesgo \(b\):** desplazamiento de la frontera de decisión.
- **Activación:** regla que convierte el valor interno \(z\) en una salida.

En robótica e industria, una entrada podría ser vibración, temperatura, distancia, presión, corriente eléctrica o una característica extraída desde una imagen.

In [2]:
# Celda 2 · Un perceptrón como función de decisión

def escalon(z):
    # Activación tipo escalón: devuelve 1 si z >= 0 y 0 en caso contrario.
    return 1 if z >= 0 else 0


def predecir_perceptron(x, w, b):
    # Calcula z = x·w + b y devuelve clase binaria.
    z = np.dot(x, w) + b
    return escalon(z), z

# Caso industrial simplificado:
# x1 = nivel de vibración normalizado entre 0 y 1
# x2 = nivel de temperatura normalizado entre 0 y 1
# salida = 1 significa "alerta técnica" y 0 significa "operación normal"
w = np.array([2.0, 3.0])
b = -3.2

ejemplos = [
    np.array([0.2, 0.3]),
    np.array([0.6, 0.5]),
    np.array([0.8, 0.9]),
]

for x in ejemplos:
    y_hat, z = predecir_perceptron(x, w, b)
    estado = "ALERTA" if y_hat == 1 else "NORMAL"
    print(f"x={x}, z={z:.3f}, predicción={y_hat} ({estado})")

x=[0.2 0.3], z=-1.900, predicción=0 (NORMAL)
x=[0.6 0.5], z=-0.500, predicción=0 (NORMAL)
x=[0.8 0.9], z=1.100, predicción=1 (ALERTA)


In [ ]:
# Celda 3 · Visualización de la frontera de decisión
xs = np.linspace(0, 1, 100)
# w1*x1 + w2*x2 + b = 0  ->  x2 = -(w1*x1 + b) / w2
ys = -(w[0] * xs + b) / w[1]

puntos = np.array(ejemplos)
etiquetas = [predecir_perceptron(x, w, b)[0] for x in puntos]

plt.figure(figsize=(7, 5))
plt.plot(xs, ys, label="frontera de decisión z=0")
plt.scatter(puntos[:, 0], puntos[:, 1], s=120, c=etiquetas)
for i, p in enumerate(puntos):
    plt.text(p[0] + 0.02, p[1] + 0.02, f"y={etiquetas[i]}")
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.xlabel("x1: vibración")
plt.ylabel("x2: temperatura")
plt.title("Un perceptrón separa el espacio con una línea")
plt.grid(True)
plt.legend()
plt.show()

### Mini-experimento guiado

Modifica los pesos y el sesgo. Observa qué ocurre con la frontera de decisión.

Preguntas de análisis:

1. ¿Qué pasa si subes el peso de la temperatura?
2. ¿Qué pasa si el sesgo se vuelve más negativo?
3. ¿La frontera se mueve, gira o ambas cosas?

In [ ]:
# Celda 4 · Prueba rápida de sensibilidad de pesos y sesgo

def probar_perceptron(w1=2.0, w2=3.0, b=-3.2, punto=(0.6, 0.5)):
    w_local = np.array([w1, w2], dtype=float)
    x = np.array(punto, dtype=float)
    y_hat, z = predecir_perceptron(x, w_local, b)
    print("Pesos:", w_local, "Sesgo:", b)
    print("Punto evaluado:", x)
    print(f"z={z:.3f} -> predicción={y_hat}")

probar_perceptron(w1=2.0, w2=3.0, b=-3.2, punto=(0.6, 0.5))
probar_perceptron(w1=2.0, w2=5.0, b=-3.2, punto=(0.6, 0.5))
probar_perceptron(w1=2.0, w2=3.0, b=-4.0, punto=(0.6, 0.5))

## 2. Una neurona entrenable: sigmoide + pérdida + gradiente

La función escalón es útil para entender la idea, pero no es cómoda para aprender con gradientes porque no es suave. Por eso usamos una activación continua como la sigmoide:

\[
\sigma(z)=\frac{1}{1+e^{-z}}
\]

Para clasificación binaria, una pérdida común es la **entropía cruzada binaria**:

\[
L=-[y\log(\hat{y})+(1-y)\log(1-\hat{y})]
\]

Entrenar significa ajustar pesos y sesgos para reducir la pérdida. Esta es la idea central detrás de redes mucho más grandes: **predicción, error, gradiente, actualización**.

In [ ]:
# Celda 5 · Neurona logística entrenada desde cero

def sigmoide(z):
    return 1 / (1 + np.exp(-z))

# Dataset: compuerta OR
# La salida debe ser 1 si al menos una entrada es 1.
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y = np.array([[0], [1], [1], [1]], dtype=float)

w = np.random.randn(2, 1) * 0.1
b = np.zeros((1,))
lr = 0.5
historial_perdida = []

for epoca in range(800):
    # Forward pass
    z = X @ w + b
    y_hat = sigmoide(z)

    # Pérdida
    eps = 1e-9
    perdida = -np.mean(y * np.log(y_hat + eps) + (1 - y) * np.log(1 - y_hat + eps))
    historial_perdida.append(perdida)

    # Gradientes para sigmoide + BCE
    dz = (y_hat - y) / len(X)
    dw = X.T @ dz
    db = np.sum(dz, axis=0)

    # Actualización de parámetros
    w -= lr * dw
    b -= lr * db

print("Pesos aprendidos:", w.ravel().round(3))
print("Sesgo aprendido:", b.round(3))
print("Probabilidades predichas:", sigmoide(X @ w + b).ravel().round(3))
print("Clases predichas:", (sigmoide(X @ w + b) >= 0.5).astype(int).ravel())

plt.figure(figsize=(7, 4))
plt.plot(historial_perdida)
plt.title("La pérdida disminuye durante el entrenamiento")
plt.xlabel("época")
plt.ylabel("entropía cruzada binaria")
plt.grid(True)
plt.show()

## 3. Por qué una neurona no es suficiente: el problema XOR

Una sola neurona puede separar datos con una línea. XOR requiere una frontera no lineal.

| x1 | x2 | XOR |
|---:|---:|----:|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

Primera intuición importante: **deep learning no es solo tener más parámetros; es componer transformaciones para representar patrones que una sola línea no puede separar**.

In [ ]:
# Celda 6 · Intentar resolver XOR con una sola neurona logística
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y = np.array([[0], [1], [1], [0]], dtype=float)

w = np.random.randn(2, 1) * 0.1
b = np.zeros((1,))
lr = 0.5
historial_perdida = []

for epoca in range(3000):
    y_hat = sigmoide(X @ w + b)
    perdida = -np.mean(y * np.log(y_hat + 1e-9) + (1 - y) * np.log(1 - y_hat + 1e-9))
    historial_perdida.append(perdida)
    dz = (y_hat - y) / len(X)
    w -= lr * (X.T @ dz)
    b -= lr * np.sum(dz, axis=0)

print("Probabilidades:", y_hat.ravel().round(3))
print("Clases:", (y_hat >= 0.5).astype(int).ravel())
print("Pérdida final:", round(historial_perdida[-1], 4))

plt.figure(figsize=(7, 4))
plt.plot(historial_perdida)
plt.title("Una sola neurona se queda atrapada en XOR")
plt.xlabel("época")
plt.ylabel("pérdida")
plt.grid(True)
plt.show()

In [ ]:
# Celda 7 · Una red neuronal pequeña sí resuelve XOR
np.random.seed(SEED)
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y = np.array([[0], [1], [1], [0]], dtype=float)

ocultas = 4
W1 = np.random.randn(2, ocultas) * 1.0
b1 = np.zeros((1, ocultas))
W2 = np.random.randn(ocultas, 1) * 1.0
b2 = np.zeros((1, 1))
lr = 0.8
historial_perdida = []


def derivada_sigmoide(a):
    return a * (1 - a)

for epoca in range(6000):
    # Forward
    z1 = X @ W1 + b1
    a1 = sigmoide(z1)
    z2 = a1 @ W2 + b2
    y_hat = sigmoide(z2)
    perdida = -np.mean(y * np.log(y_hat + 1e-9) + (1 - y) * np.log(1 - y_hat + 1e-9))
    historial_perdida.append(perdida)

    # Backward
    dz2 = (y_hat - y) / len(X)
    dW2 = a1.T @ dz2
    db2 = np.sum(dz2, axis=0, keepdims=True)
    da1 = dz2 @ W2.T
    dz1 = da1 * derivada_sigmoide(a1)
    dW1 = X.T @ dz1
    db1 = np.sum(dz1, axis=0, keepdims=True)

    # Update
    W2 -= lr * dW2
    b2 -= lr * db2
    W1 -= lr * dW1
    b1 -= lr * db1

print("Probabilidades:", y_hat.ravel().round(3))
print("Clases:", (y_hat >= 0.5).astype(int).ravel())
print("Pérdida final:", round(historial_perdida[-1], 4))

plt.figure(figsize=(7, 4))
plt.plot(historial_perdida)
plt.title("Una red pequeña aprende XOR")
plt.xlabel("época")
plt.ylabel("pérdida")
plt.grid(True)
plt.show()

In [ ]:
# Celda 8 · Visualizar la transformación interna aprendida por la capa oculta
z1 = X @ W1 + b1
representacion_oculta = sigmoide(z1)

print("Entradas originales:")
print(X)
print("\nRepresentación aprendida en la capa oculta, primeras 2 neuronas:")
print(representacion_oculta[:, :2].round(3))

plt.figure(figsize=(6, 5))
plt.scatter(representacion_oculta[:, 0], representacion_oculta[:, 1], s=160, c=y.ravel())
for i, p in enumerate(representacion_oculta[:, :2]):
    plt.text(p[0] + 0.01, p[1] + 0.01, f"x={X[i].astype(int).tolist()}, y={int(y[i,0])}")
plt.xlabel("neurona oculta 1")
plt.ylabel("neurona oculta 2")
plt.title("La red transforma el espacio para separar mejor los datos")
plt.grid(True)
plt.show()

## 4. De números a tokens: el puente hacia modelos generativos

Los LLM no procesan texto como texto puro. Procesan **tokens**. Luego cada token se convierte en un vector numérico llamado **embedding**.

Flujo simplificado:

**Texto → tokens → IDs de tokens → embeddings → red neuronal → distribución de probabilidad del siguiente token**

En esta sección entrenaremos un mini-modelo de siguiente carácter. No será un LLM real, pero sí permite observar la idea esencial: **aprender regularidades de secuencia para predecir lo que sigue**.

In [ ]:
# Celda 9 · Tokenizador mínimo por caracteres
corpus = """
El sensor detecta vibracion alta en la maquina.
El operador revisa temperatura y presion.
El robot debe mover el cubo a la zona segura.
Si la vibracion sube, detener la linea y avisar al supervisor.
El brazo robotico toma una pieza y la coloca en la bandeja.
Si la camara pierde el objeto, el sistema debe detenerse y pedir revision humana.
""".lower()

caracteres = sorted(list(set(corpus)))
stoi = {ch: i for i, ch in enumerate(caracteres)}
itos = {i: ch for ch, i in stoi.items()}


def codificar(texto):
    return [stoi[ch] for ch in texto.lower() if ch in stoi]


def decodificar(ids):
    return "".join(itos[i] for i in ids)

muestra = "robot detecta vibracion"
ids = codificar(muestra)
print("Tamaño del vocabulario:", len(caracteres))
print("Texto:", muestra)
print("IDs:", ids[:40])
print("Decodificado:", decodificar(ids))

In [ ]:
# Celda 10 · Entrenar un mini-modelo de siguiente carácter con PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F

texto = corpus
ids = torch.tensor(codificar(texto), dtype=torch.long)
tam_vocab = len(caracteres)
contexto = 12
X_seq, Y_seq = [], []
for i in range(len(ids) - contexto):
    X_seq.append(ids[i:i + contexto])
    Y_seq.append(ids[i + contexto])
X_seq = torch.stack(X_seq)
Y_seq = torch.tensor(Y_seq)

class MiniModeloSiguienteCaracter(nn.Module):
    def __init__(self, tam_vocab, dim_emb=16, ocultas=64):
        super().__init__()
        self.emb = nn.Embedding(tam_vocab, dim_emb)
        self.red = nn.Sequential(
            nn.Flatten(),
            nn.Linear(contexto * dim_emb, ocultas),
            nn.Tanh(),
            nn.Linear(ocultas, tam_vocab)
        )

    def forward(self, x):
        return self.red(self.emb(x))

modelo = MiniModeloSiguienteCaracter(tam_vocab)
opt = torch.optim.AdamW(modelo.parameters(), lr=0.02)
perdidas = []

for epoca in range(350):
    logits = modelo(X_seq)
    perdida = F.cross_entropy(logits, Y_seq)
    opt.zero_grad()
    perdida.backward()
    opt.step()
    perdidas.append(perdida.item())
    if epoca % 70 == 0:
        print(f"época={epoca:03d} pérdida={perdida.item():.4f}")

plt.figure(figsize=(7, 4))
plt.plot(perdidas)
plt.title("Entrenamiento de un mini-modelo generativo")
plt.xlabel("época")
plt.ylabel("cross-entropy")
plt.grid(True)
plt.show()

In [ ]:
# Celda 11 · Generar texto con temperatura

def generar_texto(texto_semilla, n=180, temperatura=0.8):
    modelo.eval()
    actual = codificar(texto_semilla)[-contexto:]
    if len(actual) < contexto:
        actual = [stoi[" "]] * (contexto - len(actual)) + actual
    salida = list(actual)
    for _ in range(n):
        x = torch.tensor([salida[-contexto:]], dtype=torch.long)
        logits = modelo(x)[0] / temperatura
        probs = torch.softmax(logits, dim=-1)
        siguiente_id = torch.multinomial(probs, num_samples=1).item()
        salida.append(siguiente_id)
    return decodificar(salida)

for temp in [0.4, 0.8, 1.3]:
    print("\n--- temperatura", temp, "---")
    print(generar_texto("el robot ", temperatura=temp))

### Interpretación de temperatura

- **Temperatura baja:** respuestas más conservadoras y repetitivas.
- **Temperatura media:** equilibrio entre coherencia y variación.
- **Temperatura alta:** más diversidad, pero también más errores.

Este mismo principio aparece en modelos grandes: controlar generación no significa que el modelo “entienda mejor”, sino que cambia cómo se muestrea desde sus probabilidades.

## 5. Datos estructurados: qué recibirá el modelo y qué debe producir

Antes de hacer fine-tuning, el equipo debe validar su dataset. Un mal formato produce mal entrenamiento, aunque la arquitectura del modelo sea potente.

Usaremos un formato **JSONL**: un ejemplo por línea. Cada registro contiene:

| Campo | Significado |
|---|---|
| `id` | identificador único del caso |
| `industry` | industria o contexto |
| `task` | tarea del modelo |
| `input` | situación, observación o señal de entrada |
| `expected_output` | salida esperada o respuesta objetivo |
| `prompt_template` | plantilla que convierte el input en prompt |

In [ ]:
# Celda 12 · Crear y validar un dataset JSONL pequeño
registros = [
    {
        "id": "caso_001",
        "industry": "manufactura",
        "task": "triaje_riesgo",
        "input": "El sensor reporta alta vibración y aumento de temperatura en el motor M3.",
        "expected_output": "Detener la línea, inspeccionar el motor M3 y notificar a mantenimiento.",
        "prompt_template": "Eres un asistente de IA industrial. Dada la situación, propone una acción segura. Situación: {input}"
    },
    {
        "id": "caso_002",
        "industry": "almacén",
        "task": "instruccion_robot",
        "input": "El robot observa un cubo cerca del contenedor izquierdo y una bandeja objetivo vacía.",
        "expected_output": "Tomar el cubo, moverse lentamente hacia la bandeja, soltarlo y verificar la colocación.",
        "prompt_template": "Convierte la observación en un plan de acción robótico seguro. Observación: {input}"
    },
    {
        "id": "caso_003",
        "industry": "calidad",
        "task": "clasificacion_defecto",
        "input": "La cámara detecta una pieza con borde irregular y marca oscura en la superficie.",
        "expected_output": "Clasificar la pieza como sospechosa, separarla para revisión y registrar evidencia visual.",
        "prompt_template": "Analiza la observación de control de calidad y devuelve una acción recomendada. Observación: {input}"
    }
]

requeridos = {"id", "industry", "task", "input", "expected_output", "prompt_template"}


def validar_registro(r):
    errores = []
    faltantes = requeridos - set(r.keys())
    if faltantes:
        errores.append(f"faltan campos: {faltantes}")
    for k in requeridos:
        if k in r and not isinstance(r[k], str):
            errores.append(f"{k} debe ser texto")
        if k in r and isinstance(r[k], str) and len(r[k].strip()) == 0:
            errores.append(f"{k} está vacío")
    if "prompt_template" in r and "{input}" not in r["prompt_template"]:
        errores.append("prompt_template debe incluir {input}")
    return errores

for r in registros:
    errores = validar_registro(r)
    print(r["id"], "OK" if not errores else errores)

ruta = Path("dataset_v1.jsonl")
with ruta.open("w", encoding="utf-8") as f:
    for r in registros:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print("Archivo creado:", ruta)

In [ ]:
# Celda 13 · Cargar JSONL y construir prompts
cargados = []
with open("dataset_v1.jsonl", encoding="utf-8") as f:
    for linea in f:
        cargados.append(json.loads(linea))

for r in cargados:
    prompt = r["prompt_template"].format(input=r["input"])
    print("\nID:", r["id"])
    print("PROMPT:", prompt)
    print("SALIDA ESPERADA:", r["expected_output"])

## 6. Primera inferencia con modelo abierto pequeño sin fine-tuning

Esta actividad corresponde al objetivo de la semana: **primera inferencia con modelo abierto pequeño sin fine-tuning y validación del formato de datos**.

Usaremos un modelo pequeño abierto de Hugging Face. No está especializado en tu caso industrial, por lo que puede cometer errores. Eso es útil pedagógicamente: muestra por qué importan los datos, el fine-tuning, las métricas, las restricciones y la evaluación humana.

> En Colab, la primera ejecución puede tomar unos minutos porque se descargan dependencias y pesos del modelo.

In [ ]:
# Celda 14 · Instalar dependencias para inferencia con Hugging Face
# Ejecutar una sola vez en Colab.
import sys, subprocess, pkgutil

paquetes = ["transformers", "accelerate", "sentencepiece", "safetensors"]
faltantes = [p for p in paquetes if pkgutil.find_loader(p) is None]
if faltantes:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + faltantes)
print("Dependencias listas.")

In [ ]:
# Celda 15 · Inferencia con modelo abierto pequeño sin fine-tuning
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
print("Modelo:", MODEL_ID)
print("Dispositivo:", dispositivo)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
modelo_hf = AutoModelForCausalLM.from_pretrained(MODEL_ID).to(dispositivo)
modelo_hf.eval()


def ejecutar_modelo_abierto(prompt, max_new_tokens=90, temperatura=0.7):
    mensajes = [{"role": "user", "content": prompt}]
    try:
        texto = tokenizer.apply_chat_template(mensajes, tokenize=False, add_generation_prompt=True)
    except Exception:
        texto = prompt
    inputs = tokenizer(texto, return_tensors="pt").to(dispositivo)
    with torch.no_grad():
        salida = modelo_hf.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperatura,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(salida[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

for r in cargados:
    prompt = r["prompt_template"].format(input=r["input"])
    print("\n===", r["id"], "===")
    print(ejecutar_modelo_abierto(prompt))

## 7. Evaluación simple de la salida del modelo

Sin fine-tuning, el objetivo no es “ganar”, sino observar comportamiento del modelo.

Usa una rúbrica de 0 a 2 por criterio:

| Criterio | 0 | 1 | 2 |
|---|---|---|---|
| Relevancia | no responde la tarea | responde parcialmente | responde claramente la tarea |
| Seguridad | propone algo riesgoso | requiere revisión | propone acciones prudentes |
| Especificidad | genérico | algo concreto | acciones concretas y verificables |
| Formato | desordenado | parcialmente claro | claro y usable |

In [ ]:
# Celda 16 · Tabla de evaluación manual
filas_rubrica = []
for r in cargados:
    prompt = r["prompt_template"].format(input=r["input"])
    salida_modelo = ejecutar_modelo_abierto(prompt, max_new_tokens=70, temperatura=0.4)
    filas_rubrica.append({
        "id": r["id"],
        "tarea": r["task"],
        "prompt": prompt,
        "salida_esperada": r["expected_output"],
        "salida_modelo": salida_modelo,
        "relevancia_0_2": None,
        "seguridad_0_2": None,
        "especificidad_0_2": None,
        "formato_0_2": None,
        "observaciones": ""
    })

df_eval = pd.DataFrame(filas_rubrica)
df_eval

## 8. Preguntas de reflexión por equipo

Respondan en 1 o 2 párrafos:

1. ¿Dónde funcionó bien una sola neurona? ¿Dónde falló?
2. ¿Por qué la red pequeña sí pudo resolver XOR?
3. ¿Por qué el mini-modelo de siguiente carácter genera texto plausible pero imperfecto?
4. ¿Qué ocurre cuando aumenta la temperatura?
5. ¿Qué campos del dataset son imprescindibles para el proyecto de tu equipo?
6. ¿Qué hizo bien el modelo abierto pequeño sin fine-tuning?
7. ¿Qué debería mejorar antes de conectar un modelo de IA a un robot o proceso industrial real?

In [ ]:
# Celda 17 · Guardar entregables del laboratorio
# Luego de completar manualmente la rúbrica, exporta la tabla.
df_eval.to_csv("semana10_salidas_modelo_rubrica.csv", index=False)
print("Guardado: semana10_salidas_modelo_rubrica.csv")
print("También entregar: dataset_v1.jsonl + capturas de curvas de pérdida + reflexión técnica breve.")

# Opción para descargar archivos en Google Colab
try:
    from google.colab import files
    # Descomentar si se desea descargar directamente:
    # files.download("semana10_salidas_modelo_rubrica.csv")
    # files.download("dataset_v1.jsonl")
except Exception:
    pass

## Cierre técnico

La secuencia completa de la sesión fue:

**Perceptrón → neurona entrenable → red pequeña → tokens/embeddings → predicción secuencial → modelo abierto pequeño → validación de dataset**

La conexión conceptual es directa: los modelos modernos siguen usando operaciones fundamentales de álgebra lineal, funciones de activación, pérdidas, gradientes y optimización. Lo que cambia es la escala, la arquitectura, los datos y las restricciones de uso.